# 🚛 Concrete Mixer Truck Detection System
## Production-Ready YOLO26n-OBB on Google Colab

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/YOUR_USERNAME/Truck-concreate/blob/main/Concrete_Mixer_Detection_Colab.ipynb)

---

### 📋 Overview

**Complete Pipeline:**
1. 📦 Setup Environment
2. 📊 Data Preparation & Fusion
3. 🎓 Model Training with MLflow
4. ✅ Model Validation
5. 🎬 Video Processing
6. 📈 Results Visualization

**Features:**
- YOLO26n-OBB for oriented bounding boxes
- Optical Flow + SSIM rotation detection
- State machine (POURING/IN_TRANSIT/IDLE)
- MLflow experiment tracking

---

### ⚙️ Requirements
- GPU Runtime (T4 recommended)
- ~10GB disk space
- Dataset uploaded to Google Drive or Colab

## 📦 Phase 0: Environment Setup

Clone repository and install dependencies

In [ ]:
# Check GPU
!nvidia-smi

In [ ]:
# Clone repository
!git clone https://github.com/YOUR_USERNAME/Truck-concreate.git
%cd Truck-concreate

In [ ]:
# Install dependencies
!pip install -q -r requirements.txt

print("✅ Dependencies installed!")

In [ ]:
# Mount Google Drive (if datasets are stored there)
from google.colab import drive
drive.mount('/content/drive')

# Optional: Copy datasets from Drive
# !cp -r /content/drive/MyDrive/datasets/Mixer.yolo26.zip ./data/
# !cp -r /content/drive/MyDrive/datasets/concrete\ mixed\ truck.yolo26.zip ./data/

## 📊 Phase 1: Data Preparation

Load and merge datasets using production modules

In [ ]:
# Import modules
import sys
sys.path.insert(0, './src')

from config import get_config
from data_loader import DataLoader

print("✅ Modules imported successfully!")

In [ ]:
# Load configuration
config = get_config(yaml_path='config.yaml')

print(f"📋 Project: {config['project'].PROJECT_NAME}")
print(f"🎯 Model: {config['model'].MODEL_NAME}")
print(f"📊 Classes: {config['dataset'].CLASSES}")
print(f"🔄 Batch Size: {config['model'].BATCH_SIZE}")
print(f"📈 Epochs: {config['model'].EPOCHS}")

In [ ]:
# Initialize data loader
loader = DataLoader(config['dataset'])

# Load all datasets
dataset1, dataset2 = loader.load_all_datasets()

print(f"\n✅ Datasets loaded!")
print(f"   Dataset 1: {'✓' if dataset1 else '✗'}")
print(f"   Dataset 2: {'✓' if dataset2 else '✗'}")

In [ ]:
# Merge datasets
merged_dir = loader.merge_datasets(
    [dataset1, dataset2],
    config['dataset'].MERGED_DIR
)

# Create data.yaml
data_yaml_path = loader.create_data_yaml(
    merged_dir,
    config['dataset'].CLASSES,
    config['dataset'].NUM_CLASSES
)

print(f"\n✅ Data preparation complete!")
print(f"   Merged dataset: {merged_dir}")
print(f"   Data YAML: {data_yaml_path}")

In [ ]:
# Visualize dataset statistics
from pathlib import Path
import matplotlib.pyplot as plt

train_images = list((merged_dir / 'train' / 'images').glob('*'))
train_labels = list((merged_dir / 'train' / 'labels').glob('*.txt'))

print(f"📊 Dataset Statistics:")
print(f"   Training images: {len(train_images)}")
print(f"   Training labels: {len(train_labels)}")

# Plot
fig, ax = plt.subplots(figsize=(8, 4))
categories = ['Images', 'Labels']
counts = [len(train_images), len(train_labels)]
ax.bar(categories, counts, color=['#3498db', '#2ecc71'])
ax.set_ylabel('Count')
ax.set_title('Training Dataset Statistics')
for i, v in enumerate(counts):
    ax.text(i, v + 10, str(v), ha='center', fontweight='bold')
plt.tight_layout()
plt.show()

## 🎓 Phase 2: Model Training

Train YOLO26n-OBB with MLflow tracking

In [ ]:
# Import trainer
from trainer import ModelTrainer

# Initialize trainer
trainer = ModelTrainer(
    config['model'],
    config['mlflow'],
    config['paths']
)

print("✅ Trainer initialized!")

In [ ]:
# Setup MLflow
trainer.setup_mlflow()

print("✅ MLflow tracking configured!")

In [ ]:
# Load model
model = trainer.load_model()

if model:
    print("✅ Model loaded successfully!")
    print(f"   Model type: {type(model)}")
else:
    print("❌ Failed to load model")

In [ ]:
# Train model
print("🎓 Starting training...")
print("   This may take several hours depending on GPU")
print("   Monitor progress below\n")

best_model_path, final_metrics = trainer.train(str(data_yaml_path))

print("\n✅ Training complete!")
print(f"   Best model: {best_model_path}")
print(f"   Final metrics: {final_metrics}")

In [ ]:
# Visualize training results
import pandas as pd
from pathlib import Path

# Load results
results_path = Path(config['paths'].RUNS_DIR) / config['paths'].MODEL_NAME / 'results.csv'

if results_path.exists():
    df = pd.read_csv(results_path)
    df.columns = df.columns.str.strip()
    
    # Plot metrics
    fig, axes = plt.subplots(2, 2, figsize=(15, 10))
    
    # mAP50
    if 'metrics/mAP50(B)' in df.columns:
        axes[0, 0].plot(df['epoch'], df['metrics/mAP50(B)'], 'b-', linewidth=2)
        axes[0, 0].set_title('mAP50', fontsize=12, fontweight='bold')
        axes[0, 0].set_xlabel('Epoch')
        axes[0, 0].set_ylabel('mAP50')
        axes[0, 0].grid(True, alpha=0.3)
    
    # mAP50-95
    if 'metrics/mAP50-95(B)' in df.columns:
        axes[0, 1].plot(df['epoch'], df['metrics/mAP50-95(B)'], 'g-', linewidth=2)
        axes[0, 1].set_title('mAP50-95', fontsize=12, fontweight='bold')
        axes[0, 1].set_xlabel('Epoch')
        axes[0, 1].set_ylabel('mAP50-95')
        axes[0, 1].grid(True, alpha=0.3)
    
    # Box Loss
    if 'train/box_loss' in df.columns:
        axes[1, 0].plot(df['epoch'], df['train/box_loss'], 'r-', linewidth=2)
        axes[1, 0].set_title('Box Loss', fontsize=12, fontweight='bold')
        axes[1, 0].set_xlabel('Epoch')
        axes[1, 0].set_ylabel('Loss')
        axes[1, 0].grid(True, alpha=0.3)
    
    # Class Loss
    if 'train/cls_loss' in df.columns:
        axes[1, 1].plot(df['epoch'], df['train/cls_loss'], 'orange', linewidth=2)
        axes[1, 1].set_title('Class Loss', fontsize=12, fontweight='bold')
        axes[1, 1].set_xlabel('Epoch')
        axes[1, 1].set_ylabel('Loss')
        axes[1, 1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    print("📊 Training metrics visualized!")
else:
    print("⚠️ Results file not found")

## ✅ Phase 3: Model Validation

Validate trained model on test set

In [ ]:
# Validate model
print("✅ Running validation...\n")

val_results = trainer.validate(best_model_path, str(data_yaml_path))

print("\n✅ Validation complete!")

## 🎬 Phase 4: Video Processing

Process video with rotation detection and state classification

In [ ]:
# Import video processor
from video_processor import VideoProcessor

# Initialize processor
processor = VideoProcessor(best_model_path, config['rotation'])

print("✅ Video processor initialized!")

In [ ]:
# Upload test video
from google.colab import files

print("📤 Upload your test video:")
uploaded = files.upload()

# Get uploaded filename
input_video = list(uploaded.keys())[0]
print(f"\n✅ Uploaded: {input_video}")

In [ ]:
# Process video
output_video = 'output_analysis.mp4'

print(f"🎬 Processing video: {input_video}")
print("   This may take a while...\n")

stats = processor.process_video(input_video, output_video)

print("\n✅ Video processing complete!")
print(f"   Output: {output_video}")

In [ ]:
# Visualize statistics
if stats:
    # Plot state distribution
    fig, ax = plt.subplots(figsize=(10, 6))
    
    states = list(stats.keys())
    counts = list(stats.values())
    colors = ['#e74c3c', '#f39c12', '#2ecc71', '#95a5a6', '#ecf0f1']
    
    bars = ax.bar(states, counts, color=colors[:len(states)])
    ax.set_ylabel('Frame Count', fontsize=12)
    ax.set_title('Truck State Distribution', fontsize=14, fontweight='bold')
    ax.tick_params(axis='x', rotation=45)
    
    # Add value labels
    for bar in bars:
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., height,
                f'{int(height)}',
                ha='center', va='bottom', fontweight='bold')
    
    plt.tight_layout()
    plt.show()
    
    # Print summary
    total_frames = sum(stats.values())
    print("\n📊 State Summary:")
    for state, count in stats.items():
        percentage = (count / total_frames) * 100 if total_frames > 0 else 0
        print(f"   {state}: {count} frames ({percentage:.1f}%)")

In [ ]:
# Download processed video
from google.colab import files

print("📥 Downloading processed video...")
files.download(output_video)
print("✅ Download started!")

## 📈 Phase 5: MLflow Results

View experiment tracking results

In [ ]:
# Start MLflow UI (in background)
import subprocess
import time

# Start MLflow server
mlflow_process = subprocess.Popen(
    ['mlflow', 'ui', '--backend-store-uri', './mlruns', '--port', '5000'],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE
)

time.sleep(3)
print("✅ MLflow UI started!")
print("\n⚠️ Note: MLflow UI cannot be accessed directly in Colab")
print("   Download mlruns/ folder and view locally")

In [ ]:
# List MLflow experiments
import mlflow

mlflow.set_tracking_uri('file://./mlruns')

experiments = mlflow.search_experiments()
print("📊 MLflow Experiments:\n")
for exp in experiments:
    print(f"   Name: {exp.name}")
    print(f"   ID: {exp.experiment_id}")
    print(f"   Artifact Location: {exp.artifact_location}")
    print()

In [ ]:
# Download MLflow artifacts
!zip -r mlruns.zip mlruns/

print("📦 Downloading MLflow artifacts...")
files.download('mlruns.zip')
print("✅ Download started!")

## 📦 Phase 6: Export & Download

Export model and download all artifacts

In [ ]:
# Export to ONNX
from ultralytics import YOLO

print("📦 Exporting model to ONNX...")

model = YOLO(best_model_path)
onnx_path = model.export(format='onnx', dynamic=True, simplify=True)

print(f"\n✅ ONNX export complete!")
print(f"   Path: {onnx_path}")

In [ ]:
# Download trained model
print("📥 Downloading trained model...")
files.download(best_model_path)
print("✅ Model download started!")

In [ ]:
# Create deployment package
import shutil
from pathlib import Path

# Create deployment directory
deploy_dir = Path('deployment_package')
deploy_dir.mkdir(exist_ok=True)

# Copy files
shutil.copy(best_model_path, deploy_dir / 'best.pt')
if Path(onnx_path).exists():
    shutil.copy(onnx_path, deploy_dir / 'model.onnx')
shutil.copy('config.yaml', deploy_dir / 'config.yaml')
shutil.copy('requirements.txt', deploy_dir / 'requirements.txt')

# Zip deployment package
!zip -r deployment_package.zip deployment_package/

print("📦 Deployment package created!")
print("\n📥 Downloading deployment package...")
files.download('deployment_package.zip')
print("✅ Download started!")

## 🎉 Summary

### ✅ Completed Tasks:

1. ✅ **Environment Setup** - Dependencies installed
2. ✅ **Data Preparation** - Datasets loaded and merged
3. ✅ **Model Training** - YOLO26n-OBB trained with MLflow
4. ✅ **Validation** - Model performance evaluated
5. ✅ **Video Processing** - Rotation detection and state classification
6. ✅ **Export** - ONNX model and deployment package

### 📊 Outputs:

- `best.pt` - Trained PyTorch model
- `model.onnx` - ONNX exported model
- `output_analysis.mp4` - Processed video
- `mlruns.zip` - MLflow experiment data
- `deployment_package.zip` - Complete deployment package

### 🚀 Next Steps:

1. Review MLflow metrics locally
2. Test ONNX model on edge devices
3. Deploy to production environment
4. Monitor real-time performance

---

**🎓 Model:** YOLO26n-OBB  
**📊 Framework:** Ultralytics + MLflow  
**🔄 Features:** Rotation Detection + State Machine  
**🌐 Deployment:** ONNX Runtime Ready  

---

**Version:** 1.0.0  
**Last Updated:** 2026-04-14